In [1]:
import numpy as np 
import pandas as pd

In [2]:
match=pd.read_csv("matches.csv")
deliveries=pd.read_csv("deliveries.csv")

In [3]:
match.sample(2)

,id,Season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
675,7933,IPL-2018,Jaipur,08-05-2018,Rajasthan Royals,Kings XI Punjab,Rajasthan Royals,bat,normal,0,Rajasthan Royals,15,0,JC Buttler,Sawai Mansingh Stadium,Marais Erasmus,Nitin Menon,Yeshwant Barde
42,43,IPL-2017,Hyderabad,06-05-2017,Rising Pune Supergiant,Sunrisers Hyderabad,Sunrisers Hyderabad,field,normal,0,Rising Pune Supergiant,12,0,JD Unadkat,"Rajiv Gandhi International Stadium, Uppal",KN Ananthapadmanabhan,AK Chaudhary,NaN


In [4]:
deliveries.sample(2)

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
93622,396,1,Royal Challengers Bangalore,Chennai Super Kings,4,4,V Kohli,MA Agarwal,MM Sharma,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
42116,179,2,Chennai Super Kings,Deccan Chargers,6,2,S Badrinath,JM Kemp,PP Ojha,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN


In [5]:
total_score_df=deliveries.groupby(["match_id","inning"]).sum()["total_runs"].reset_index()

In [6]:
total_score_df=total_score_df[total_score_df["inning"]==1]

In [7]:
matchdf=match.merge(total_score_df[["match_id","total_runs"]],left_on="id",right_on="match_id")

In [8]:
matchdf["team1"].unique()

array(['Sunrisers Hyderabad', 'Mumbai Indians', 'Gujarat Lions',
       'Rising Pune Supergiant', 'Royal Challengers Bangalore',
       'Kolkata Knight Riders', 'Delhi Daredevils', 'Kings XI Punjab',
       'Chennai Super Kings', 'Rajasthan Royals', 'Deccan Chargers',
       'Kochi Tuskers Kerala', 'Pune Warriors', 'Rising Pune Supergiants',
       'Delhi Capitals'], dtype=object)

In [9]:
teams = ['Sunrisers Hyderabad', 
         'Mumbai Indians', 
         'Royal Challengers Bangalore',
         'Kolkata Knight Riders',  
         'Kings XI Punjab',
         'Chennai Super Kings', 
         'Rajasthan Royals', 
         'Delhi Capitals']

In [10]:
matchdf["team1"]=matchdf["team1"].str.replace('Delhi Daredevils','Delhi Capitals')
matchdf["team2"]=matchdf["team2"].str.replace('Delhi Daredevils','Delhi Capitals')

matchdf["team1"]=matchdf["team1"].str.replace('Deccan Chargers','Sunrisers Hyderabad')
matchdf["team2"]=matchdf["team2"].str.replace('Deccan Chargers','Sunrisers Hyderabad')

In [11]:
matchdf = matchdf[matchdf["team1"].isin(teams)]
matchdf = matchdf[matchdf["team2"].isin(teams)]

In [12]:
matchdf.shape

(641, 20)

In [13]:
matchdf=matchdf[matchdf["dl_applied"] == 0]

In [14]:
matchdf=matchdf[["match_id","city","winner","total_runs"]]

In [15]:
deldf=matchdf.merge(deliveries,on="match_id")

In [16]:
deldf=deldf[deldf["inning"]==2]

In [17]:
deldf.shape

(72413, 24)

In [18]:
deldf["current_score"]=deldf.groupby("match_id").cumsum()["total_runs_y"]

In [19]:
deldf["runs_left"]=deldf["total_runs_x"] - deldf["current_score"]

In [20]:
deldf

,match_id,city,winner,total_runs_x,inning,batting_team,bowling_team,over,ball,batsman,...,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs_y,player_dismissed,dismissal_kind,fielder,current_score,runs_left
125,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,CH Gayle,...,0,0,1,0,1,NaN,NaN,NaN,1,206
126,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,Mandeep Singh,...,0,0,0,0,0,NaN,NaN,NaN,1,206
127,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,Mandeep Singh,...,0,0,0,0,0,NaN,NaN,NaN,1,206
128,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,Mandeep Singh,...,0,0,2,0,2,NaN,NaN,NaN,3,204
129,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,Mandeep Singh,...,0,0,4,0,4,NaN,NaN,NaN,7,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149573,11415,Hyderabad,Mumbai Indians,152,2,Chennai Super Kings,Mumbai Indians,20,2,RA Jadeja,...,0,0,1,0,1,NaN,NaN,NaN,152,0
149574,11415,Hyderabad,Mumbai Indians,152,2,Chennai Super Kings,Mumbai Indians,20,3,SR Watson,...,0,0,2,0,2,NaN,NaN,NaN,154,-2
149575,11415,Hyderabad,Mumbai Indians,152,2,Chennai Super Kings,Mumbai Indians,20,4,SR Watson,...,0,0,1,0,1,SR Watson,run out,KH Pandya,155,-3
149576,11415,Hyderabad,Mumbai Indians,152,2,Chennai Super Kings,Mumbai Indians,20,5,SN Thakur,...,0,0,2,0,2,NaN,NaN,NaN,157,-5


In [21]:
deldf["balls_left"]=126-(deldf["over"]*6 + deldf["ball"])

In [22]:
deldf["player_dismissed"]=deldf["player_dismissed"].fillna("0")
deldf["player_dismissed"]=deldf["player_dismissed"].apply(lambda x:x if x == "0" else "1")
deldf["player_dismissed"]=deldf["player_dismissed"].astype("int")
wickets=deldf.groupby("match_id").cumsum()["player_dismissed"].values
deldf["wickets"]=10 - wickets
deldf.head(3)

,match_id,city,winner,total_runs_x,inning,batting_team,bowling_team,over,ball,batsman,...,batsman_runs,extra_runs,total_runs_y,player_dismissed,dismissal_kind,fielder,current_score,runs_left,balls_left,wickets
125,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,CH Gayle,...,1,0,1,0,NaN,NaN,1,206,119,10
126,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,Mandeep Singh,...,0,0,0,0,NaN,NaN,1,206,118,10
127,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,Mandeep Singh,...,0,0,0,0,NaN,NaN,1,206,117,10


In [23]:
deldf.tail()

,match_id,city,winner,total_runs_x,inning,batting_team,bowling_team,over,ball,batsman,...,batsman_runs,extra_runs,total_runs_y,player_dismissed,dismissal_kind,fielder,current_score,runs_left,balls_left,wickets
149573,11415,Hyderabad,Mumbai Indians,152,2,Chennai Super Kings,Mumbai Indians,20,2,RA Jadeja,...,1,0,1,0,NaN,NaN,152,0,4,5
149574,11415,Hyderabad,Mumbai Indians,152,2,Chennai Super Kings,Mumbai Indians,20,3,SR Watson,...,2,0,2,0,NaN,NaN,154,-2,3,5
149575,11415,Hyderabad,Mumbai Indians,152,2,Chennai Super Kings,Mumbai Indians,20,4,SR Watson,...,1,0,1,1,run out,KH Pandya,155,-3,2,4
149576,11415,Hyderabad,Mumbai Indians,152,2,Chennai Super Kings,Mumbai Indians,20,5,SN Thakur,...,2,0,2,0,NaN,NaN,157,-5,1,4
149577,11415,Hyderabad,Mumbai Indians,152,2,Chennai Super Kings,Mumbai Indians,20,6,SN Thakur,...,0,0,0,1,lbw,NaN,157,-5,0,3


In [24]:
#crr runs/overs
deldf["crr"]=(deldf["current_score"]*6)/(120-deldf["balls_left"])

In [25]:
deldf["rrr"]=(deldf["runs_left"]*6)/deldf["balls_left"]

In [26]:
deldf.head()

,match_id,city,winner,total_runs_x,inning,batting_team,bowling_team,over,ball,batsman,...,total_runs_y,player_dismissed,dismissal_kind,fielder,current_score,runs_left,balls_left,wickets,crr,rrr
125,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,CH Gayle,...,1,0,NaN,NaN,1,206,119,10,6.0,10.386555
126,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,Mandeep Singh,...,0,0,NaN,NaN,1,206,118,10,3.0,10.474576
127,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,Mandeep Singh,...,0,0,NaN,NaN,1,206,117,10,2.0,10.564103
128,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,Mandeep Singh,...,2,0,NaN,NaN,3,204,116,10,4.5,10.551724
129,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,Mandeep Singh,...,4,0,NaN,NaN,7,200,115,10,8.4,10.434783


In [27]:
def result(row):
    return 1 if row["batting_team"] == row['winner'] else 0

In [28]:
deldf["result"]=deldf.apply(result,axis=1)

In [29]:
finaldf=deldf[["batting_team",'bowling_team','city','runs_left','balls_left','wickets','total_runs_x','crr','rrr','result']]

In [30]:
finaldf=finaldf.sample(finaldf.shape[0])

In [31]:
finaldf.sample()

,batting_team,bowling_team,city,runs_left,balls_left,wickets,total_runs_x,crr,rrr,result
88888,Rajasthan Royals,Chennai Super Kings,NaN,103,90,9,140,7.4,6.866667,0


In [32]:
finaldf.dropna(inplace=True)

In [33]:
finaldf=finaldf[finaldf['balls_left'] != 0]

In [34]:
X=finaldf.iloc[:,:-1]
y=finaldf.iloc[:,-1]

In [35]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split (X,y,test_size=0.2,random_state=1)

In [36]:
X_train.shape

(57073, 9)

In [37]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
trf = ColumnTransformer([
    ('trf',OneHotEncoder(sparse=False,drop="first"),['batting_team','bowling_team','city'])
],remainder='passthrough')

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

In [39]:
pipe= Pipeline(steps=[('step1',trf),
                       ('step2',LogisticRegression(solver='liblinear'))
                     ])

In [40]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('step1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('trf',
                                                  OneHotEncoder(drop='first',
                                                                sparse=False),
                                                  ['batting_team',
                                                   'bowling_team', 'city'])])),
                ('step2', LogisticRegression(solver='liblinear'))])

In [41]:
y_pred = pipe.predict(X_test)

In [42]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.8041208213609924

In [43]:
pipe.predict_proba(X_test)[0]

array([0.20988846, 0.79011154])

In [44]:
teams

['Sunrisers Hyderabad',
 'Mumbai Indians',
 'Royal Challengers Bangalore',
 'Kolkata Knight Riders',
 'Kings XI Punjab',
 'Chennai Super Kings',
 'Rajasthan Royals',
 'Delhi Capitals']

In [45]:
deldf['city'].unique()

array(['Hyderabad', 'Bangalore', 'Mumbai', 'Indore', 'Kolkata', 'Delhi',
       'Chandigarh', 'Jaipur', 'Chennai', 'Cape Town', 'Port Elizabeth',
       'Durban', 'Centurion', 'East London', 'Johannesburg', 'Kimberley',
       'Bloemfontein', 'Ahmedabad', 'Cuttack', 'Nagpur', 'Dharamsala',
       'Visakhapatnam', 'Pune', 'Raipur', 'Ranchi', 'Abu Dhabi',
       'Sharjah', nan, 'Mohali', 'Bengaluru'], dtype=object)

In [46]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))